In [1]:
import re
from declaration import VariableDeclaration, HybridProgramDeclaration, PredicateDeclaration
from KyxParser import KyxParser

In [2]:
kyx_path = 'D:/Reinforcement learning/kyx_parser/kyxs/CoastingCar_with_pred.kyx'
kyx_folder_path = 'D:/Reinforcement learning/kyx_parser/kyxs'
template = 'D:/Reinforcement learning/kyx_parser/env_template.py'

In [3]:
def read_file_as_string(filepath):
    """Reads the entire file into a single string.

    Args:
        filepath: The path to the text file.

    Returns:
        The file content as a string, or None if the file doesn't exist.
    """
    try:
        with open(filepath, 'r') as file:
            file_content = file.read()
            return file_content
    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return None

# Example usage:
kyx_content = read_file_as_string(kyx_path)
template_content = read_file_as_string(template)

In [4]:
print(kyx_content)

ArchiveEntry "Coasting Car"
Description "Coasting cars can stop".

Definitions       /* function symbols cannot change their value */
  Real b;         /* braking capabilities */
  Real ep;        /* reaction time */

  /* precondition */
  Bool precondition(Real v, Real m, Real x) <-> speedConstraint(v, m, x) &  v >= 0 & b > 0;
  
  Bool speedConstraint(Real v, Real m, Real x) <-> v^2<=2*b*(m-x);

  /* controller */
  HP controller ::= {a:=-b; ++ ?v^2<=2*b*(m-x-ep*v); a:=0;};

  /* step */
  HP stepFunction ::= {x'=v, v'=a, t'=1 & v>=0 & t<=ep};

  /* done */
  Bool doneCondition(Real x, Real m) <-> x <= m;

End.

ProgramVariables  /* program variables may change their value over time */
  Real x;         /* position of the car */
  Real v;         /* velocity of the car */
  Real a;         /* acceleration of the car */
  Real m;         /* stop sign position */
  Real t;         /* clock variable */
End.

Problem           /* differential dynamic logic formula */
    precondition(v,

In [5]:
# Example usage:
parser = KyxParser()
parsed_data = parser.parse(kyx_content) # kyx_content is your KYX file content
print(parsed_data)

Python Param:  float v, float m, float x
Python Param:  float v, float m, float x
Python Param:  float x, float m
res:  {'Definitions': [<declaration.VariableDeclaration object at 0x0000014099A54D70>, <declaration.VariableDeclaration object at 0x000001409985F750>, <declaration.PredicateDeclaration object at 0x0000014099A54EC0>, <declaration.PredicateDeclaration object at 0x000001409985FC50>, <declaration.HybridProgramDeclaration object at 0x0000014099A55010>, <declaration.HybridProgramDeclaration object at 0x000001409985FD90>, <declaration.PredicateDeclaration object at 0x000001409985FED0>], 'ProgramVariables': [<declaration.VariableDeclaration object at 0x0000014099A88050>, <declaration.VariableDeclaration object at 0x00000140999F9350>, <declaration.VariableDeclaration object at 0x00000140999F90F0>, <declaration.VariableDeclaration object at 0x0000014099A3DA30>, <declaration.VariableDeclaration object at 0x0000014099A11BF0>], 'Problem': {'precondition': 'precondition(v, m, x)', 'model

In [6]:
import re

def _parse_declarations(self, declarations_string):
    """Parses declaration strings (Definitions, ProgramVariables) into a dictionary.
       Handles more complex formats including comments and function definitions.
    """
    declarations = {}
    lines = declarations_string.splitlines()
    in_definition_block = False  # Flag to track if we're inside a definition block

    for line in lines:
        line = line.strip()
        if line.startswith("Definitions"):
            in_definition_block = True
            continue
        elif line == "End.":
            in_definition_block = False
            continue
        elif not line or line.startswith("/*"):  # Ignore empty lines and comments
            continue

        if in_definition_block:
            # Use regex to find declarations within the definition block
            match = re.match(r'\s*(\w+)\s+(\w+);', line) # Matches "type name;"
            if match:
                var_type, var_name = match.groups()
                declarations[var_name] = var_type
            else:
                # Handle cases where the declaration format is not "type name;"
                print(f"Warning: Could not parse declaration: {line}")

    return declarations

# Example Usage:
declarations_string = """
Definitions       /* function symbols cannot change their value */
  Real b;         /* braking capabilities */
  Real ep;        /* reaction time */

  /* precondition */
  Bool precondition(Real v, Real m, Real x) <-> speedConstraint(v, m, x) &  v >= 0 & b > 0;

  Bool speedConstraint(Real v, Real m, Real x) <-> v^2<=2*b*(m-x);

  /* controller */
  HP controller(Real v, Real m, Real x) ::= {a:=-b; ++ ?v^2<=2*b*(m-x-ep*v); a:=0;};

  /* step */
  HP stepFunction(Real v, Real a, Real t) ::= {x'=v, v'=a, t'=1 & v>=0 & t<=ep};

  /* done */
  Bool doneCondition(Real x, Real m) <-> x <= m;

End.
"""

declarations = _parse_declarations(None, declarations_string)  # Assuming 'self' is not needed here.
print(declarations)

{'b': 'Real', 'ep': 'Real'}


In [7]:
import os

def iterate_files_top_level(directory):
    """Iterates over files in the specified directory only (not recursively)."""
    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        if os.path.isfile(filepath):  # Check if it's a file (not a directory)
            print(f"Processing file: {filepath}")
            # Add your file processing logic here

            content = read_file_as_string(filepath)
            parser = KyxParser()
            parsed_data = parser.parse(content) # kyx_content is your KYX file content
            print(parsed_data)


# Example usage:
iterate_files_top_level(kyx_folder_path)

Processing file: D:/Reinforcement learning/kyx_parser/kyxs\carExc.kyx
Error: Could not parse the problem statement. Check the format.
res:  {'Definitions': [<declaration.VariableDeclaration object at 0x0000014099A11480>, <declaration.VariableDeclaration object at 0x0000014099875050>, <declaration.VariableDeclaration object at 0x0000014099874250>], 'ProgramVariables': [<declaration.VariableDeclaration object at 0x000001409986C5F0>, <declaration.VariableDeclaration object at 0x000001409986DB80>, <declaration.VariableDeclaration object at 0x0000014099A68F30>, <declaration.VariableDeclaration object at 0x0000014099A68C90>, <declaration.VariableDeclaration object at 0x0000014099A77110>], 'Problem': {'error': 'Could not parse problem statement'}}
float b
float A
float ep
{'Definitions': [<declaration.VariableDeclaration object at 0x0000014099A11480>, <declaration.VariableDeclaration object at 0x0000014099875050>, <declaration.VariableDeclaration object at 0x0000014099874250>], 'ProgramVariab

In [8]:
print(kyx_content)

ArchiveEntry "Coasting Car"
Description "Coasting cars can stop".

Definitions       /* function symbols cannot change their value */
  Real b;         /* braking capabilities */
  Real ep;        /* reaction time */

  /* precondition */
  Bool precondition(Real v, Real m, Real x) <-> speedConstraint(v, m, x) &  v >= 0 & b > 0;
  
  Bool speedConstraint(Real v, Real m, Real x) <-> v^2<=2*b*(m-x);

  /* controller */
  HP controller ::= {a:=-b; ++ ?v^2<=2*b*(m-x-ep*v); a:=0;};

  /* step */
  HP stepFunction ::= {x'=v, v'=a, t'=1 & v>=0 & t<=ep};

  /* done */
  Bool doneCondition(Real x, Real m) <-> x <= m;

End.

ProgramVariables  /* program variables may change their value over time */
  Real x;         /* position of the car */
  Real v;         /* velocity of the car */
  Real a;         /* acceleration of the car */
  Real m;         /* stop sign position */
  Real t;         /* clock variable */
End.

Problem           /* differential dynamic logic formula */
    precondition(v,

In [9]:
kyx_parser(kyx_content)

NameError: name 'kyx_parser' is not defined